# 🎬 PeakCut Studio: One-Click Google Colab Runner
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreeharsh-patil/PeakCut/blob/main/PeakCut_Colab.ipynb)

**Turn long videos and live streams into viral 9:16 vertical shorts** using NVIDIA GPU acceleration.

### ⚡ Instructions:
1. Set Hardware Accelerator to GPU: **Runtime** → **Change runtime type** → **T4 GPU**.
2. (Optional) Enter your Gemini API key (or multiple comma-separated keys for auto-failover).
3. **Click the Run button (▶)** on the code cell below.
4. Once the setup finishes, click the generated **`https://*.trycloudflare.com`** link to open PeakCut Studio in your browser!

In [ ]:
# @title 🚀 Run PeakCut Studio (One-Click Start)
# @markdown ### 🔑 Gemini API Keys (Optional, Comma-Separated)
# @markdown Enter one or multiple Gemini API keys. If multiple keys are provided, PeakCut will automatically rotate keys when rate limits or quotas are reached. Leave blank to run with the built-in heuristic virality engine.
GEMINI_API_KEYS = "" # @param {type:"string"}

# @markdown ### 🎙️ Speech-to-Text Model
WHISPER_MODEL = "base" # @param ["tiny", "base", "small", "medium", "large-v3"]

import os, subprocess, time, re

if GEMINI_API_KEYS.strip():
    os.environ["GEMINI_API_KEYS"] = GEMINI_API_KEYS.strip()
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEYS.strip()

os.environ["WHISPER_MODEL"] = WHISPER_MODEL
os.environ["WHISPER_DEVICE"] = "cuda"
os.environ["WHISPER_COMPUTE"] = "float16"

# 1. System packages & hardware tools
print("📦 Step 1/5: Installing system dependencies (ffmpeg, fonts)...")
subprocess.run(
    "apt-get update -qq && apt-get install -y -qq ffmpeg libsm6 libxext6 libgl1 fonts-freefont-ttf fonts-dejavu-core",
    shell=True, check=True
)

# 2. Download Cloudflared for instant zero-config public HTTPS tunnel
print("🌐 Step 2/5: Setting up Cloudflare tunnel binary...")
subprocess.run(
    "wget -q -nc -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared",
    shell=True, check=True
)

# 3. Clone or update repository from GitHub
REPO_URL = "https://github.com/shreeharsh-patil/PeakCut.git"
REPO_DIR = "/content/PeakCut"

print(f"📥 Step 3/5: Fetching repository from {REPO_URL}...")
if not os.path.exists(REPO_DIR):
    subprocess.run(f"git clone {REPO_URL} {REPO_DIR}", shell=True, check=True)
else:
    subprocess.run(f"git -C {REPO_DIR} pull", shell=True)

os.chdir(REPO_DIR)

# 4. Install Python dependencies
print("🐍 Step 4/5: Installing Python dependencies...")
subprocess.run("pip install -q -r requirements.txt", shell=True)
subprocess.run("pip install -q uvicorn fastapi requests", shell=True)

# 5. Start Cloudflare Tunnel in background
print("🚀 Step 5/5: Starting Cloudflare tunnel & web server...")
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

# 6. Extract and display the public HTTPS link
public_url = None
for _ in range(40):
    line = tunnel.stdout.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

print("\n" + "=" * 70)
if public_url:
    print(f"🎉 PEAKCUT STUDIO IS LIVE AT: {public_url}")
    print("👉 Click the link above to open PeakCut Studio in your browser!")
else:
    print("⚠️ Tunnel starting... check output below for the link.")
print("=" * 70 + "\n")

# 7. Start FastAPI Server with PeakCut Studio UI
subprocess.run("python -m uvicorn app:app --host 127.0.0.1 --port 8000", shell=True)
